In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Results
## generating table

In [ ]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results/ablations/distributions/structural_baseline/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
    if 'structure_baseline' in method:
        method_group[method] = 'Baseline'
    elif 'divcurves' in method:
        method_group[method] = 'Diversity Curves'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
        'divcurves_shortest_path_distance': 'Shortest Path',
        'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
        'structure_baseline_nnodes': 'Number of Nodes',
        'structure_baseline_nedges': 'Number of Edges',
        'structure_baseline_density': 'Density',
        'structure_baseline_nconn_components': 'Number of Connected Components',
        'structure_baseline_avg_clustering': 'Clustering Coefficient',
        'structure_baseline_summary': 'Summary Statistic'
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
         &                 &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.904 $\pm $ 0.042} &           0.427 $\pm $ 0.034 &           0.748 $\pm $ 0.060 &             194.340 $\pm $ 34.081 &           0.511 $\pm $ 0.081 &           0.311 $\pm $ 0.119 \\
         & Shortest Path PCA &           0.888 $\pm $ 0.053 &           0.436 $\pm $ 0.034 &           0.732 $\pm $ 0.060 &             197.355 $\pm $ 35.008 &           0.511 $\p

In [14]:
df

silh_mean    db_mean  \
Group            Method                                                 
Diversity Curves Shortest Path                    0.426791   0.748132   
                 Shortest Path PCA                0.435842   0.732327   
Baseline         Density                          0.546073   0.616190   
                 Summary Statistic               -0.079592   3.291008   
                 Clustering Coefficient           0.041178   4.252141   
                 Number of Edges                 -0.093871   3.665604   
                 Number of Connected Components  -0.133582   1.712764   
                 Number of Nodes                 -0.083137  99.298258   

                                                     ch_mean  silh_std  \
Group            Method                                                  
Diversity Curves Shortest Path                    194.340472  0.033794   
                 Shortest Path PCA                197.354888  0.034473   
Baseline         Density                         1769.541061  0.037596   
                 Summary Statistic                 33.052206  0.019914   
                 Clustering Coefficient            34.702434  0.030873   
                 Number of Edges                   33.508424  0.021170   
                 Number of Connected Components    68.197326  0.055083   
                 Number of Nodes                    0.531599  0.012990   

                                                     db_std      ch_std  \
Group            Method                                                   
Diversity Curves Shortest Path                     0.059847   34.081011   
                 Shortest Path PCA                 0.059935   35.007875   
Baseline         Density                           0.086067  289.570143   
                 Summary Statistic                 1.143852    5.790938   
                 Clustering Coefficient            4.872266    5.472647   
                 Number of Edges                   2.337651    5.866155   
                 Number of Connected Components    0.408686   14.380464   
                 Number of Nodes                 196.612996    0.473977   

                                                 ari_agglo_mean  \
Group            Method                                           
Diversity Curves Shortest Path                         0.310997   
                 Shortest Path PCA                     0.317161   
Baseline         Density                               0.633415   
                 Summary Statistic                     0.088472   
                 Clustering Coefficient                0.129107   
                 Number of Edges                       0.088511   
                 Number of Connected Components        0.185879   
                 Number of Nodes                      -0.011531   

                                                 ari_agglo_std  \
Group            Method                                          
Diversity Curves Shortest Path                        0.119196   
                 Shortest Path PCA                    0.122657   
Baseline         Density                              0.049325   
                 Summary Statistic                    0.030140   
                 Clustering Coefficient               0.090779   
                 Number of Edges                      0.029497   
                 Number of Connected Components       0.032038   
                 Number of Nodes                      0.006669   

                                                 ari_kmeans_mean  \
Group            Method                                            
Diversity Curves Shortest Path                          0.511371   
                 Shortest Path PCA                      0.510657   
Baseline         Density                                0.729381   
                 Summary Statistic                      0.110025   
                 Clustering Coefficient                 0.234923   
                 N